In [ ]:
# 1. SETUP

!pip install transformers datasets wordcloud --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

from wordcloud import WordCloud

import re
import string

np.random.seed(42)


In [ ]:
# 2. LOAD DATA


from google.colab import files
import io

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='latin-1')

print("Columns in dataset:", df.columns)

if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    df.columns = ['label', 'text']

elif 'label' in df.columns and 'text' in df.columns:
    df = df[['label', 'text']]

else:
    df = df.iloc[:, :2]
    df.columns = ['label', 'text']

df['label'] = df['label'].astype(str).str.lower()

df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1,
    '0': 0,
    '1': 1
})

df = df.dropna()

print("\nCleaned dataset preview:")
print(df.head())

print("\nClass distribution:")
print(df['label'].value_counts())

In [ ]:
# 3. EDA + VISUALIZATION

df['length'] = df['text'].apply(len)

plt.figure()
sns.histplot(data=df, x='length', hue='label', bins=50)
plt.title("Text Length Distribution")
plt.show()

spam_words = ' '.join(df[df['label']==1]['text'])
ham_words = ' '.join(df[df['label']==0]['text'])

plt.figure(figsize=(10,5))
plt.imshow(WordCloud().generate(spam_words))
plt.title("Spam Word Cloud")
plt.axis('off')
plt.show()

plt.figure(figsize=(10,5))
plt.imshow(WordCloud().generate(ham_words))
plt.title("Ham Word Cloud")
plt.axis('off')
plt.show()

In [ ]:
# 4. PREPROCESSING

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
# 5. TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

In [ ]:
# 6. BASELINE MODELS

nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('model', MultinomialNB())
])

lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('model', LogisticRegression(max_iter=200))
])

nb_pipeline.fit(X_train, y_train)
lr_pipeline.fit(X_train, y_train)

In [ ]:
# 7. EVALUATION FUNCTION

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    print(f"\n{name} RESULTS")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d')
    plt.title(f"{name} Confusion Matrix")
    plt.show()

    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC_AUC': roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    }

In [ ]:
# 8. RUN EVALUATION

nb_results = evaluate_model("Naive Bayes", nb_pipeline, X_test, y_test)
lr_results = evaluate_model("Logistic Regression", lr_pipeline, X_test, y_test)

In [ ]:
# 9. CROSS VALIDATION

cv_nb = cross_val_score(nb_pipeline, X_train, y_train, cv=5)
cv_lr = cross_val_score(lr_pipeline, X_train, y_train, cv=5)

print("NB CV Accuracy:", cv_nb.mean())
print("LR CV Accuracy:", cv_lr.mean())

In [ ]:
# 10. RULE-BASED LAYER

spam_keywords = ['win', 'free', 'cash', 'prize', 'urgent', 'claim']

def rule_based_filter(text):
    for word in spam_keywords:
        if word in text:
            return 1
    return None

def hybrid_predict(text):
    rule = rule_based_filter(text)
    if rule is not None:
        return rule
    return lr_pipeline.predict([text])[0]

# Apply hybrid
y_pred_hybrid = [hybrid_predict(x) for x in X_test]

print("\nHYBRID MODEL")
print(classification_report(y_test, y_pred_hybrid))

In [ ]:
# ==============================
# 11. MODEL COMPARISON TABLE
# ==============================

results_df = pd.DataFrame([nb_results, lr_results])
results_df

In [ ]:
# 12. TRANSFORMER MODEL

from transformers import pipeline

classifier = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english")

sample_texts = X_test[:200]

preds = []
for text in sample_texts:
    result = classifier(text[:512])[0]
    preds.append(1 if result['label']=='POSITIVE' else 0)

print("Transformer sample predictions done")

In [ ]:
# 13. EMAIL PREDICTION FUNCTION

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

def predict_email(text):
    global tokenizer
    if 'tokenizer' not in globals():
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    global model
    if 'model' not in globals():
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        spam_prob = probs[0][1].item()

    return "Spam" if spam_prob >= 0.5 else "Not Spam"

In [ ]:
# 14. FINAL RANDOM EMAIL TEST
import random

random_index = random.randint(0, len(df)-1)

sample_email = df.iloc[random_index]['text']
actual_label = "Spam" if df.iloc[random_index]['label'] == 1 else "Not Spam"

predicted_label = predict_email(sample_email)

print("EMAIL:")

print(sample_email)
print("\nACTUAL:", actual_label)
print("PREDICTED:", predicted_label)